# Phase A — data collection

Multi-season, 22-division match+odds corpus (items 1, 2, 4) plus the ClubElo
join check (item 3).

`QUICK_TEST = True` for a fast few-season run, `False` for the full range.


## Setup

In [1]:
# pip install pandas requests pyarrow
import csv
import datetime
import os
import time
import warnings
from collections import Counter

import pandas as pd
import requests

pd.set_option("display.width", 160)
warnings.filterwarnings("ignore", message="DataFrame is highly fragmented")


In [2]:
DIVISIONS = [
    "E0", "E1", "E2", "E3", "EC",          # England
    "SC0", "SC1", "SC2", "SC3",            # Scotland
    "D1", "D2",                            # Germany
    "SP1", "SP2",                          # Spain
    "I1", "I2",                            # Italy
    "F1", "F2",                            # France
    "N1", "B1", "P1", "T1", "G1",          # NL, BE, PT, TR, GR (top flight)
]

BASE_URL = "https://www.football-data.co.uk/mmz4281"
FIRST_SEASON_YEAR = 2005   # older seasons are thin on odds coverage anyway

# Pinnacle feed has been broken since this date (confirmed in EDA.py: PSH
# exceeds the recorded market max in ~25% of matches from here on, coverage
# drops to 39.3%). Mask by date, not by season.
PINNACLE_STALE_FROM = "2025-07-23"

CACHE_DIR = "fd_cache"
OUT_PARQUET = "matches_multiseason.parquet"

# football-data renamed this club mid-corpus. Left alone, our Elo (keyed on
# division + team) would start the "new" club at 1500 in 2026/27. Only put
# confirmed renames of the SAME club here -- the validation section reports
# candidates, but similar names can be different clubs (Reggina/Reggiana).
TEAM_CANONICAL = {
    "AFC Telford United": "Telford United",
    "Atl. Madrid": "Ath Madrid",
}

# football-data spelling -> ClubElo spelling. Used only for the join; the
# match data keeps football-data's own names.
CLUBELO_ALIASES = {
    "Ankaragucu": "Ankaraguecue",
    "Extremadura UD": "Extremadura",
    "FeralpiSalo": "Feralpisalo",
    "Inverness C": "Inverness",
    "Kallithea": "Kalithea",
    "Kifisia": "Kifisias",
    "Sheffield Wed": "Sheffield Weds",
}

# Reviewed and rejected. Without these the same two false positives surface
# on every run, and a report you learn to ignore is worse than no report.
CLUBELO_NOT_A_MATCH = {
    "Northwich",                      # Northwich Victoria is not Norwich City
}
NOT_RENAMES = {
    ("Reggina", "Reggiana"),          # different cities; both played 2020-2023
}


In [3]:
def season_code(start_year):
    # 2025 -> '2526', 1999 -> '9900'
    return f"{start_year % 100:02d}{(start_year + 1) % 100:02d}"


def default_seasons(first_year=FIRST_SEASON_YEAR, today=None):
    today = today or datetime.date.today()
    last_year = today.year if today.month >= 7 else today.year - 1
    return [season_code(y) for y in range(first_year, last_year + 1)]


print(season_code(2025), season_code(1999))
print("seasons since", FIRST_SEASON_YEAR, "->", len(default_seasons()))


2526 9900
seasons since 2005 -> 22


## Date parsing

Export eras differ (day-first vs month-first). Fit the format per file, before
concatenating — one format across a mixed corpus mis-parses whichever era
loses the vote.


In [4]:
_DATE_FORMATS = ("%d/%m/%Y", "%d/%m/%y", "%m/%d/%Y", "%m/%d/%y", "%Y-%m-%d")

# which format won for each (season, div) -- audited in the validation section
DATE_FORMAT_LOG = {}


def parse_dates(s, key=None):
    s = s.astype(str).str.strip()
    valid = s.ne("") & s.ne("nan")
    best, best_hits = None, -1
    for fmt in _DATE_FORMATS:
        hits = pd.to_datetime(s.where(valid), format=fmt, errors="coerce").notna().sum()
        if hits > best_hits:
            best, best_hits = fmt, hits
        if hits == valid.sum():
            break
    if key is not None:
        DATE_FORMAT_LOG[key] = (best, int(best_hits), int(valid.sum()))
    return pd.to_datetime(s.where(valid), format=best, errors="coerce")


## Fetch

In [5]:
# default python-requests UA gets blocked by some hosts
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-data-fetch/1.0)"}


def _cache_path(cache_dir, season, div):
    return os.path.join(cache_dir, season, f"{div}.csv")


def fetch_one(session, season, div, cache_dir, timeout=15):
    # status string rather than a bool: a 404 and a dead network need
    # different handling and shouldn't collapse into "skipped"
    path = _cache_path(cache_dir, season, div)
    if os.path.exists(path):
        return "cached"

    try:
        r = session.get(f"{BASE_URL}/{season}/{div}.csv", headers=HEADERS, timeout=timeout)
    except requests.Timeout:
        return "timeout"
    except requests.RequestException as e:
        return f"neterr:{type(e).__name__}"

    if r.status_code != 200:
        return f"http{r.status_code}"
    if len(r.content) < 500:
        return "tiny"
    # block pages come back 200 with plenty of bytes -- don't cache them as csv
    if b"HomeTeam" not in r.content[:1000]:
        return "notcsv"

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as fh:
        fh.write(r.content)
    return "ok"


def fetch_seasons(seasons, divisions, cache_dir=CACHE_DIR, polite_delay=0.15,
                  abort_after=8):
    session = requests.Session()
    tally, consecutive_net_fail = Counter(), 0

    for season in seasons:
        per = Counter()
        for div in divisions:
            status = fetch_one(session, season, div, cache_dir)
            per[status] += 1
            tally[status] += 1

            # bail early rather than burn 20 minutes of timeouts
            if status == "timeout" or status.startswith("neterr"):
                consecutive_net_fail += 1
                if consecutive_net_fail >= abort_after:
                    raise RuntimeError(
                        f"{consecutive_net_fail} consecutive network failures — aborting"
                    )
            else:
                consecutive_net_fail = 0

            if status == "ok":
                time.sleep(polite_delay)

        print(f"{season}: {dict(per)}", flush=True)

    return tally


In [6]:
QUICK_TEST = False   # False = fetch everything since FIRST_SEASON_YEAR

seasons = default_seasons(first_year=2022) if QUICK_TEST else default_seasons()
print(f"{len(seasons)} seasons x {len(DIVISIONS)} divisions: {seasons}")


22 seasons x 22 divisions: ['0506', '0607', '0708', '0809', '0910', '1011', '1112', '1213', '1314', '1415', '1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425', '2526', '2627']


In [7]:
tally = fetch_seasons(seasons, DIVISIONS, CACHE_DIR)
print("\ntotals:", dict(tally))

# any http* = no file published for that season/division, expected for
# seasons not yet played
usable = tally["ok"] + tally["cached"]
absent = sum(v for k, v in tally.items() if k.startswith("http"))
print(f"{usable} files usable, {absent} not published yet")


0506: {'cached': 22}
0607: {'cached': 22}
0708: {'cached': 22}
0809: {'cached': 22}
0910: {'cached': 22}
1011: {'cached': 22}
1112: {'cached': 22}
1213: {'cached': 22}
1314: {'cached': 22}
1415: {'cached': 22}
1516: {'cached': 22}
1617: {'cached': 22}
1718: {'cached': 22}
1819: {'cached': 22}
1920: {'cached': 22}
2021: {'cached': 22}
2122: {'cached': 22}
2223: {'cached': 22}
2324: {'cached': 22}
2425: {'cached': 22}
2526: {'cached': 22}
2627: {'http300': 6, 'cached': 16}

totals: {'cached': 478, 'http300': 6}
478 files usable, 6 not published yet


## Assembly

A single junk token (`#REF!`, a stray space) makes pandas read that whole
file's column as strings. Concatenated against the same column read as float
elsewhere it becomes a mixed object column: parquet rejects it, and comparisons
against it are silently wrong. Coerce after concat, log what changed.


In [8]:
# columns that are legitimately text and must never be coerced
NON_NUMERIC = {"Div", "Date", "Season", "Time", "HomeTeam", "AwayTeam",
               "FTR", "HTR", "Referee", "Country"}


def coerce_numeric(df, report=True):
    fixed = []
    for c in df.columns:
        if c in NON_NUMERIC or df[c].dtype != object:
            continue
        nonnull = int(df[c].notna().sum())
        if not nonnull:
            continue
        conv = pd.to_numeric(df[c], errors="coerce")
        got = int(conv.notna().sum())
        # only adopt if the column really is numeric; a text column converts
        # almost entirely to NaN
        if got >= 0.9 * nonnull:
            fixed.append((c, nonnull, nonnull - got))
            df[c] = conv

    if report and fixed:
        print(f"coerced {len(fixed)} object columns to numeric:")
        for c, n, lost in fixed:
            note = f"  ({lost} unparseable -> NaN)" if lost else ""
            print(f"  {c:<12} {n:>8,} values{note}")
    return df


def mixed_type_columns(df, sample=20000):
    # anything still mixed dies in to_parquet with an unreadable pyarrow
    # error; name the column here instead
    bad = []
    for c in df.columns:
        if c in NON_NUMERIC or df[c].dtype != object:
            continue
        kinds = {type(v).__name__ for v in df[c].dropna().head(sample)}
        if len(kinds) > 1:
            bad.append((c, sorted(kinds)))
    return bad


In [9]:
def _read_any_encoding(path, **kw):
    try:
        return pd.read_csv(path, encoding="utf-8", low_memory=False, **kw)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False, **kw)


def _scan_fields(path):
    for enc in ("utf-8", "latin-1"):
        try:
            with open(path, encoding=enc, newline="") as fh:
                rows = csv.reader(fh)
                header = next(rows)
                return header, max([len(header)] + [len(r) for r in rows if r])
        except UnicodeDecodeError:
            continue
    raise ValueError(f"cannot decode {path}")


def _read_cached(path, season, div):
    try:
        df = _read_any_encoding(path)
    except pd.errors.ParserError:
        # football-data appends columns without extending the header row, so
        # later rows carry more fields than the header declares. Pad to the
        # widest row; skipping them drops real matches. header=None +
        # skiprows=1 so the field count comes from `names`, not the short
        # header line.
        header, widest = _scan_fields(path)
        names = header + [f"Extra{i}" for i in range(len(header), widest)]
        df = _read_any_encoding(path, names=names, header=None, skiprows=1)
        print(f"  {season}/{div}: ragged file, header padded {len(header)} -> {widest}")

    # old files pad team names with spaces; "Ajax " and "Ajax" would be two
    # different clubs to both our Elo and the ClubElo join
    for c in ("HomeTeam", "AwayTeam"):
        if c in df.columns:
            df[c] = df[c].str.strip()
    if "HomeTeam" in df.columns:
        df = df.dropna(subset=["HomeTeam"])
    df["Season"], df["Div"] = season, div
    if "Date" in df.columns:
        df["Date"] = parse_dates(df["Date"], key=(season, div))
        df = df.dropna(subset=["Date"])
    return df


def load_matches(seasons, divisions, cache_dir=CACHE_DIR):
    frames = []
    for season in seasons:
        for div in divisions:
            path = _cache_path(cache_dir, season, div)
            if os.path.exists(path):
                frames.append(_read_cached(path, season, div))

    if not frames:
        raise FileNotFoundError(f"nothing cached in {cache_dir!r} -- run fetch_seasons() first")

    df = pd.concat(frames, ignore_index=True, sort=False)
    df = coerce_numeric(df)
    for c in ("HomeTeam", "AwayTeam"):
        df[c] = df[c].replace(TEAM_CANONICAL)
    return df.sort_values("Date").reset_index(drop=True)


In [10]:
df = load_matches(seasons, DIVISIONS, CACHE_DIR)
print(f"{len(df):,} matches, {df.Div.nunique()} divisions, "
      f"{df.Date.min().date()} to {df.Date.max().date()}")
df.head(3)


  0607/T1: ragged file, header padded 55 -> 63
  0708/SP2: ragged file, header padded 58 -> 61
  0708/F2: ragged file, header padded 58 -> 61
  0708/N1: ragged file, header padded 58 -> 61
  0708/P1: ragged file, header padded 58 -> 61
coerced 7 object columns to numeric:
  BbAH          108,439 values  (4 unparseable -> NaN)
  BbAHh         108,438 values  (10 unparseable -> NaN)
  PSCH          101,470 values
  B365CH         53,497 values  (1 unparseable -> NaN)
  BWCA           49,985 values  (1 unparseable -> NaN)
  AvgCAHH        53,527 values  (1 unparseable -> NaN)
  1XBH            7,507 values  (1 unparseable -> NaN)
162,447 matches, 22 divisions, 2005-07-29 to 2026-08-20


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,PPA,SKBH,SKBD,SKBA,PPCH,PPCD,PPCA,SKBCH,SKBCD,SKBCA
0,F2,2005-07-29,Amiens,Grenoble,3.0,0.0,H,1.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F2,2005-07-29,Valenciennes,Dijon,2.0,1.0,H,2.0,1.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F2,2005-07-29,Reims,Lorient,1.0,2.0,A,0.0,1.0,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pinnacle masking

Blank `PS*` from `PINNACLE_STALE_FROM` onward. Not a season-wide drop —
Pinnacle is the sharpest book in the file before the feed broke.


In [11]:
# PS* (1X2), PSC* (closing), P>/P< (O/U), PAH* (handicap) -- no other
# bookmaker code in this schema starts with "P".
_PINNACLE_PREFIXES = ("PS", "PC", "P>", "P<", "PAH")


def mask_stale_pinnacle(df, cutoff=PINNACLE_STALE_FROM):
    pinnacle_cols = [c for c in df.columns if c.startswith(_PINNACLE_PREFIXES)]
    if not pinnacle_cols:
        return df
    mask = df["Date"] >= pd.Timestamp(cutoff)
    df.loc[mask, pinnacle_cols] = pd.NA
    return df


df = mask_stale_pinnacle(df)

if "PSH" in df.columns:
    after = df[df.Date >= PINNACLE_STALE_FROM]
    before = df[df.Date < PINNACLE_STALE_FROM]
    print(f"after cutoff: PSH non-null = {after.PSH.notna().sum()} / {len(after)} (want 0)")
    print(f"before cutoff: PSH non-null = {before.PSH.notna().sum()} / {len(before)}")


after cutoff: PSH non-null = 0 / 7875 (want 0)
before cutoff: PSH non-null = 98944 / 154572


In [12]:
# drop matches with no result or no opening 1X2 price
df = df.dropna(subset=["FTR"])
odds_cols = [c for c in ("B365H", "B365D", "B365A") if c in df.columns]
if odds_cols:
    df = df.dropna(subset=odds_cols)
df = df.sort_values("Date").reset_index(drop=True)

bad = mixed_type_columns(df)
if bad:
    print("mixed-type columns parquet will reject:")
    for c, kinds in bad:
        print(f"  {c}: {kinds}")
    raise TypeError("fix the columns above before writing")

df.to_parquet(OUT_PARQUET, index=False)
print(f"{len(df):,} usable matches -> {OUT_PARQUET}")


162,053 usable matches -> matches_multiseason.parquet


## Coverage table

Per-season completeness by column group. This is what fixes the modelling
window, not the changelog.


In [13]:
def coverage_table(df):
    groups = {
        "Result": ["FTHG", "FTAG", "FTR"],
        "Officials": ["Referee"],
        "Shots": ["HS", "AS", "HST", "AST"],
        "Discipline": ["HF", "AF", "HY", "AY", "HR", "AR"],
        "Corners": ["HC", "AC"],
        "1X2 opening": ["B365H", "B365D", "B365A"],
        "1X2 closing": ["B365CH", "B365CD", "B365CA"],
        "Pinnacle opening": ["PSH", "PSD", "PSA"],
        "Market best": ["MaxH", "MaxD", "MaxA"],
        "Market avg": ["AvgH", "AvgD", "AvgA"],
        "Asian handicap": ["AHh", "B365AHH", "B365AHA"],
    }
    seasons = sorted(df["Season"].unique())

    rows = []
    for name, cols in groups.items():
        cols = [c for c in cols if c in df.columns]
        if not cols:
            continue
        for season in seasons:
            g = df.loc[df["Season"] == season, cols]
            cov = g.notna().all(axis=1).mean() if len(g) else float("nan")
            rows.append({"Season": season, "Group": name, "Coverage": cov})

    table = (pd.DataFrame(rows)
             .pivot(index="Season", columns="Group", values="Coverage")
             .reindex(seasons))

    # flag groups swinging between usable (>=80%) and unusable (<20%)
    # mid-sample; report the span, since a column can die as well as arrive
    flags = {}
    for name in table.columns:
        s = table[name].dropna()
        usable = s[s >= 0.8]
        if len(s) < 2 or usable.empty or not (s < 0.2).any():
            continue
        lo, hi = usable.index.min(), usable.index.max()
        notes = []
        if lo > s.index.min():
            notes.append(f"absent before {lo}")
        if hi < s.index.max():
            notes.append(f"absent after {hi}")
        flags[name] = f"usable {lo}-{hi}" + (f" ({'; '.join(notes)})" if notes else "")

    return table, flags


table, flags = coverage_table(df)
display(table.round(3))
print("flags:", flags or "none (widen QUICK_TEST range to see this properly)")


Group,1X2 closing,1X2 opening,Asian handicap,Corners,Discipline,Market avg,Market best,Officials,Pinnacle opening,Result,Shots
Season,,,,,,,,,,,
0506,0.000,1.0,0.000,0.487,0.487,0.0,0.0,0.399,0.000,1.0,0.496
0607,0.000,1.0,0.000,0.585,0.537,0.0,0.0,0.409,0.000,1.0,0.585
0708,0.000,1.0,0.000,0.546,0.546,0.0,0.0,0.361,0.000,1.0,0.546
0809,0.000,1.0,0.000,0.545,0.545,0.0,0.0,0.360,0.000,1.0,0.545
0910,0.000,1.0,0.000,0.554,0.554,0.0,0.0,0.364,0.000,1.0,0.554
1011,0.000,1.0,0.000,0.551,0.551,0.0,0.0,0.364,0.000,1.0,0.551
1112,0.000,1.0,0.000,0.549,0.549,0.0,0.0,0.361,0.000,1.0,0.549
1213,0.000,1.0,0.000,0.551,0.551,0.0,0.0,0.049,0.979,1.0,0.551
1314,0.000,1.0,0.000,0.547,0.547,0.0,0.0,0.361,0.976,1.0,0.547


flags: {'1X2 closing': 'usable 1920-2627 (absent before 1920)', 'Asian handicap': 'usable 1920-2627 (absent before 1920)', 'Market avg': 'usable 1920-2627 (absent before 1920)', 'Market best': 'usable 1920-2627 (absent before 1920)', 'Pinnacle opening': 'usable 1213-2425 (absent before 1213; absent after 2425)'}


## ClubElo join check (item 3)

xgabora already merged football-data with ClubElo and reconciled the names for
2000-2025 — the expensive part. Clone once:

```bash
git clone https://github.com/xgabora/Club-Football-Match-Data-2000-2025
```

Verify the join on a sample rather than trusting it. Coverage is uneven by
design (ClubElo tracks Europe's top ~500 only) and that unevenness is itself
a variance proxy for item 23.


In [14]:
import difflib

XGABORA_DIR = "Club-Football-Match-Data-2000-2025"

if not os.path.isdir(XGABORA_DIR):
    print(f"'{XGABORA_DIR}' not found -- clone it first (see cell above)")
else:
    elo = pd.read_csv(os.path.join(XGABORA_DIR, "data", "EloRatings.csv"))
    elo_recent = set(elo[elo.date >= "2024-08-01"].club.unique())
    elo_ever = set(elo.club.unique())

    def to_clubelo(t):
        return CLUBELO_ALIASES.get(t, t)

    rows = []
    for div, g in df.groupby("Div"):
        teams = set(pd.concat([g.HomeTeam, g.AwayTeam]).dropna().unique())
        if not teams:
            continue
        matched = sum(to_clubelo(t) in elo_recent for t in teams)
        rows.append((div, len(teams), matched, matched / len(teams) * 100))

    join_check = pd.DataFrame(rows, columns=["Div", "Teams", "MatchedInClubElo", "MatchRate%"])
    display(join_check.sort_values("MatchRate%", ascending=False).round(1))


,Div,Teams,MatchedInClubElo,MatchRate%
3,E0,44,37,84.1
1,D1,37,31,83.8
19,SP1,43,35,81.4
8,F1,41,29,70.7
11,I1,45,31,68.9
4,E1,58,37,63.8
15,SC0,19,12,63.2
13,N1,32,18,56.2
2,D2,58,32,55.2
9,F2,58,30,51.7


Split the unmatched. A club ClubElo never tracked is expected; one it tracks
under a different spelling is a broken join, and on a big club that is a
silently missing feature.


In [15]:
if os.path.isdir(XGABORA_DIR):
    all_teams = set(pd.concat([df.HomeTeam, df.AwayTeam]).dropna().unique())

    rows = []
    for t in sorted(t for t in all_teams if CLUBELO_ALIASES.get(t, t) not in elo_recent):
        if CLUBELO_ALIASES.get(t, t) in elo_ever:
            rows.append((t, "historic only", ""))       # tracked, not top-500 now
            continue
        close = difflib.get_close_matches(t, elo_ever, n=1, cutoff=0.85)
        if close and t not in CLUBELO_NOT_A_MATCH:
            rows.append((t, "NAME MISMATCH", close[0]))
        else:
            rows.append((t, "not tracked", ""))

    unmatched = pd.DataFrame(rows, columns=["Team", "Status", "ClosestInClubElo"])
    print(unmatched.Status.value_counts().to_string())

    mismatches = unmatched[unmatched.Status == "NAME MISMATCH"]
    if len(mismatches):
        print(f"\n{len(mismatches)} likely broken joins — check these by hand:")
        display(mismatches)


historic only    261
not tracked      154


`MatchRate%` above asks whether a club sits in ClubElo's *current* top 500,
which over 22 seasons counts every long-gone club as a failure. Item 23 needs
coverage at the match date, so measure per match.


In [16]:
if os.path.isdir(XGABORA_DIR):
    span = (elo.assign(date=pd.to_datetime(elo.date))
              .groupby("club").date.agg(["min", "max"]))
    lo, hi = span["min"].to_dict(), span["max"].to_dict()

    def rated(team, when):
        t = CLUBELO_ALIASES.get(team, team)
        return t in lo and lo[t] <= when <= hi[t]

    both = [rated(h, d) and rated(a, d)
            for h, a, d in zip(df.HomeTeam, df.AwayTeam, df.Date)]
    cov = pd.DataFrame({"Div": df.Div.values, "both": both})

    # static snapshot: anything after its last date has no ClubElo at all,
    # which is a coverage limit rather than a join failure
    print(f"ClubElo snapshots run {span['min'].min().date()} to {span['max'].max().date()}")
    after = (df.Date > span["max"].max()).sum()
    if after:
        print(f"{after:,} matches fall after that and can never match — "
              f"fetch api.clubelo.com directly for the current season")

    print(f"matches with a ClubElo rating for both teams at kickoff: "
          f"{cov.both.mean() * 100:.1f}%")
    per_div = cov.groupby("Div").both.agg(["mean", "size"])
    per_div["mean"] = (per_div["mean"] * 100).round(1)
    display(per_div.rename(columns={"mean": "Covered%", "size": "Matches"})
            .sort_values("Covered%", ascending=False))


ClubElo snapshots run 2000-07-01 to 2025-06-01
7,850 matches fall after that and can never match — fetch api.clubelo.com directly for the current season
matches with a ClubElo rating for both teams at kickoff: 66.5%


,Covered%,Matches
Div,,
I2,95.7,8951
F1,95.5,7654
I1,95.2,7972
SC0,94.9,4749
SP1,94.7,7987
E0,94.0,7980
D2,93.9,6440
P1,93.4,5911
F2,93.4,7738


### Team name stability across seasons

football-data renames clubs mid-corpus (`Ath Madrid` -> `Atl. Madrid`). That
breaks the ClubElo join and splits our own Elo, which is keyed on
`(division, team)` and restarts the "new" club at 1500.

Promotion and relegation churn teams legitimately, so a departure alone means
nothing. Only a departure whose name matches an arrival in the same division
the next season is a candidate.


In [17]:
# "Malaga B" after "Malaga" is a reserve side, not a rename
_RESERVE_SUFFIXES = (" B", " II", " C")


def _reserve_pair(a, b):
    short, long_ = sorted((a, b), key=len)
    return any(long_ == short + suf for suf in _RESERVE_SUFFIXES)


def suspected_renames(df, cutoff=0.85):
    out = []
    for div, g in df.groupby("Div"):
        per_season = {s: set(pd.concat([x.HomeTeam, x.AwayTeam]).dropna())
                      for s, x in g.groupby("Season")}
        seasons = sorted(per_season)
        for a, b in zip(seasons, seasons[1:]):
            gone, arrived = per_season[a] - per_season[b], per_season[b] - per_season[a]
            for t in sorted(gone):
                close = difflib.get_close_matches(t, arrived, n=1, cutoff=cutoff)
                if (close and not _reserve_pair(t, close[0])
                        and (t, close[0]) not in NOT_RENAMES):
                    out.append((div, a, b, t, close[0]))
    return pd.DataFrame(out, columns=["Div", "LastSeen", "ThenAppears", "OldName", "NewName"])


renames = suspected_renames(df)
if len(renames):
    print(f"{len(renames)} suspected renames — each would split one club's Elo history:")
    display(renames)
else:
    print("no suspected renames")


no suspected renames


## Validation

The fetch only proves the plumbing works. Each check below targets a failure
mode that a single season, or a few adjacent ones, structurally cannot
surface.


### 1. Date parsing

The riskiest silent failure: old exports use 2-digit years, new ones 4-digit,
and a day/month swap is invisible downstream. Check which format won per file,
and whether the parsed dates land in the season they are labelled with.


In [18]:
fmt = pd.DataFrame(
    [(s, d, f, hits, total) for (s, d), (f, hits, total) in DATE_FORMAT_LOG.items()],
    columns=["Season", "Div", "Format", "Parsed", "Rows"],
)

# a file where the winning format didn't parse everything is a real problem
partial = fmt[fmt.Parsed < fmt.Rows]
print(f"files where the chosen format left rows unparsed: {len(partial)}")
if len(partial):
    display(partial)

print("\nformats used per season:")
display(fmt.groupby("Season").Format.agg(lambda s: sorted(set(s))).to_frame())


files where the chosen format left rows unparsed: 0

formats used per season:


,Format
Season,
0506,[%d/%m/%y]
0607,"[%d/%m/%Y, %d/%m/%y]"
0708,[%d/%m/%y]
0809,"[%d/%m/%Y, %d/%m/%y]"
0910,[%d/%m/%y]
1011,[%d/%m/%y]
1112,[%d/%m/%y]
1213,[%d/%m/%y]
1314,[%d/%m/%y]


In [19]:
def season_start_year(code):
    yy = int(code[:2])
    return 1900 + yy if yy >= 90 else 2000 + yy


rows = []
for season, g in df.groupby("Season"):
    y = season_start_year(season)
    lo, hi = pd.Timestamp(f"{y}-07-01"), pd.Timestamp(f"{y + 1}-06-30")
    out = g.Date[(g.Date < lo) | (g.Date > hi)]
    rows.append((season, g.Date.min().date(), g.Date.max().date(), len(g), len(out),
                 out.min().date() if len(out) else "",
                 out.max().date() if len(out) else ""))

span = pd.DataFrame(rows, columns=["Season", "First", "Last", "Matches",
                                   "OutsideWindow", "OutsideFrom", "OutsideTo"])
display(span)

bad = span[span.OutsideWindow > 0]
print("seasons with dates outside their own window:", len(bad))
if len(bad):
    print("Check the OutsideFrom/To dates. A misparse scatters them randomly;")
    print("a contiguous run past June is a real calendar change — 2019/20 ran")
    print("to August 2020 because of the COVID suspension.")


,Season,First,Last,Matches,OutsideWindow,OutsideFrom,OutsideTo
0,0506,2005-07-29,2006-06-18,7764,0,,
1,0607,2006-07-28,2007-06-17,7799,0,,
2,0708,2007-07-27,2008-06-15,7803,0,,
3,0809,2008-08-01,2009-06-21,7787,0,,
4,0910,2009-07-31,2010-06-19,7610,0,,
5,1011,2010-07-30,2011-06-04,7734,0,,
6,1112,2011-07-15,2012-06-03,7698,0,,
7,1213,2012-07-27,2013-06-09,7738,0,,
8,1314,2013-07-19,2014-06-08,7796,0,,
9,1415,2014-07-25,2015-06-07,7844,0,,


seasons with dates outside their own window: 1
Check the OutsideFrom/To dates. A misparse scatters them randomly;
a contiguous run past June is a real calendar change — 2019/20 ran
to August 2020 because of the COVID suspension.


### 2. Column lifespans

A feature that only appears halfway through the corpus can't be used across it
without introducing sample selection. All-null counts as absent.


In [20]:
WATCH = ["B365H", "B365CH", "MaxH", "AvgH", "PSH", "BFEH",
         "AHh", "B365AHH", "HS", "HST", "HC", "HY", "Referee"]

rows = []
for c in WATCH:
    if c not in df.columns:
        rows.append((c, "-", "-", 0))
        continue
    s = df.groupby("Season")[c].apply(lambda x: x.notna().any())
    present = s[s].index.tolist()
    rows.append((c, present[0] if present else "-",
                 present[-1] if present else "-", int(s.sum())))

lifespan = pd.DataFrame(rows, columns=["Column", "FirstSeason", "LastSeason", "SeasonsPresent"])
display(lifespan.sort_values("SeasonsPresent"))
print(f"total seasons in corpus: {df.Season.nunique()}")


,Column,FirstSeason,LastSeason,SeasonsPresent
5,BFEH,2425,2627,3
1,B365CH,1920,2627,8
2,MaxH,1920,2627,8
3,AvgH,1920,2627,8
6,AHh,1920,2627,8
7,B365AHH,1920,2627,8
4,PSH,1213,2425,13
0,B365H,0506,2627,22
8,HS,0506,2627,22
9,HST,0506,2627,22


total seasons in corpus: 22


### 3. Pinnacle masking, both branches

Masking by date only earns its complexity if it keeps Pinnacle before the
cutoff. That branch needs a season ending before 2025-07-23 to exercise.


In [21]:
if "PSH" in df.columns:
    before = df[df.Date < PINNACLE_STALE_FROM]
    after = df[df.Date >= PINNACLE_STALE_FROM]

    kept = int(before.PSH.notna().sum())
    leaked = int(after.PSH.notna().sum())

    print(f"before cutoff: {kept:,} / {len(before):,} rows keep Pinnacle")
    print(f"after cutoff:  {leaked:,} / {len(after):,} rows still have Pinnacle (want 0)")

    if len(before) == 0:
        print("\nno rows before the cutoff — widen the season range to exercise this")
    elif kept == 0:
        print("\nPinnacle absent before the cutoff too — check the source columns")


before cutoff: 98,873 / 154,203 rows keep Pinnacle
after cutoff:  0 / 7,850 rows still have Pinnacle (want 0)


In [22]:
key = ["Div", "Date", "HomeTeam", "AwayTeam"]
dupes = df.duplicated(subset=key).sum()
print(f"duplicate matches on {key}: {dupes} (want 0)")
if dupes:
    display(df[df.duplicated(subset=key, keep=False)].sort_values(key)[key].head(20))


duplicate matches on ['Div', 'Date', 'HomeTeam', 'AwayTeam']: 0 (want 0)


### 4. Fetch gaps

A division that runs for years, vanishes for one season, then returns is a
failed download, not a format change.


In [23]:
counts = df.pivot_table(index="Div", columns="Season", values="Date",
                        aggfunc="count", fill_value=0)
display(counts)

gaps = []
for div, row in counts.iterrows():
    nz = row.to_numpy().nonzero()[0]
    if len(nz) < 2:
        continue
    for i in range(nz[0], nz[-1]):
        if row.iloc[i] == 0:
            gaps.append((div, row.index[i]))

print("interior gaps (division missing between seasons it otherwise ran):", gaps or "none")


Season,0506,0607,0708,0809,0910,1011,1112,1213,1314,1415,...,1718,1819,1920,2021,2122,2223,2324,2425,2526,2627
Div,,,,,,,,,,,,,,,,,,,,,
B1,302,304,306,306,210,240,240,240,240,240,...,240,240,232,306,305,306,312,310,308,18
D1,306,306,306,306,306,306,306,306,306,306,...,306,306,306,306,306,306,306,306,306,0
D2,306,306,306,306,306,306,306,306,306,305,...,306,306,306,303,306,306,306,306,306,18
E0,380,380,380,380,380,380,380,380,380,380,...,380,380,380,380,380,380,380,380,380,0
E1,552,552,552,552,552,552,552,552,552,552,...,552,552,552,552,552,552,552,552,552,12
E2,552,552,552,552,552,552,552,552,552,552,...,552,552,398,552,552,552,552,552,552,13
E3,552,552,552,552,552,552,552,552,552,552,...,552,552,436,552,551,551,552,552,552,12
EC,461,549,552,537,506,552,516,552,552,552,...,552,552,451,474,506,552,551,552,540,24
F1,380,380,380,380,380,380,380,380,380,380,...,380,380,279,378,380,380,306,306,306,0


interior gaps (division missing between seasons it otherwise ran): none


## Analysis windows

Closing odds are missing before 2019/20; ClubElo is missing for clubs outside
Europe's top ~500 and for everything after the dump's last date.

Decided against dropping those matches. ClubElo coverage tracks division tier
almost exactly — ~95% in the top flights, 0% in E3, EC, SC2, SC3 — so dropping
deletes the lower divisions wholesale. Those are the soft, high-margin markets
this project is about, and selecting on a variable that correlates with the
outcome is the same class of bias the study is measuring.

Carry availability as flags; each analysis restricts itself and reports its
own n. Missingness is informative anyway: no ClubElo means the club is outside
the tracked set, a direct signal that its rating is uncertain, which is what
`sigma_e^2(x)` is after in item 23.


In [24]:
CLOSING = [c for c in ("B365CH", "B365CD", "B365CA") if c in df.columns]
df["has_closing"] = df[CLOSING].notna().all(axis=1) if CLOSING else False
df["has_clubelo"] = both if os.path.isdir(XGABORA_DIR) else False

w = pd.DataFrame([
    ("full corpus", len(df)),
    ("with closing odds", int(df.has_closing.sum())),
    ("with ClubElo", int(df.has_clubelo.sum())),
    ("with both", int((df.has_closing & df.has_clubelo).sum())),
], columns=["Window", "Matches"])
w["% of corpus"] = (w.Matches / len(df) * 100).round(1)
display(w)

df.to_parquet(OUT_PARQUET, index=False)
print(f"flags added, {OUT_PARQUET} rewritten")


,Window,Matches,% of corpus
0,full corpus,162053,100.0
1,with closing odds,53387,32.9
2,with ClubElo,107733,66.5
3,with both,31892,19.7


flags added, matches_multiseason.parquet rewritten


In [25]:
# per-division cost of dropping, i.e. the reason not to
loss = (df.groupby("Div")
          .agg(Matches=("has_clubelo", "size"), Kept=("has_clubelo", "sum")))
loss["Lost%"] = ((1 - loss.Kept / loss.Matches) * 100).round(1)
display(loss.sort_values("Lost%", ascending=False))
print(f"dropping uncovered matches would remove "
      f"{int((~df.has_clubelo).sum()):,} matches "
      f"({(~df.has_clubelo).mean() * 100:.1f}%), concentrated in the lower tiers")


,Matches,Kept,Lost%
Div,,,
SC3,3680,0,100.0
SC2,3678,0,100.0
EC,11187,0,100.0
E3,11486,14,99.9
E2,11451,541,95.3
SC1,3699,318,91.4
B1,5685,4741,16.6
D1,6426,5493,14.5
G1,5054,4352,13.9


dropping uncovered matches would remove 54,320 matches (33.5%), concentrated in the lower tiers


## Next

- `QUICK_TEST = False` for the full range; most of the validation section is
  vacuous on a short one.
- Reload with `pd.read_parquet("matches_multiseason.parquet")`.
